# Jenis-jenis Klasifikasi dan Dataset-nya

Tugas Machine Learning.

Isi notebook ini:
1. Perbedaan tiga jenis klasifikasi: **binary**, **multi-class (ATAU)**, dan **multi-label (DAN)**
2. 3 dataset untuk tiap jenis, lengkap dengan narasi deskripsi dan keterangan asal dataset
3. Identifikasi jenis klasifikasi tiap dataset menggunakan **pandas**

## 1. Perbedaan Binary, Multi-class, dan Multi-label

| Aspek | Binary | Multi-class (**ATAU**) | Multi-label (**DAN**) |
|---|---|---|---|
| Jumlah kelas/label yang tersedia | Tepat **2** kelas | **Lebih dari 2** kelas | **2 label atau lebih** |
| Jumlah label yang dimiliki **satu data** | **1** (salah satu dari 2 kelas) | **1** (salah satu dari banyak kelas) | **0, 1, atau banyak** label sekaligus |
| Hubungan antar kelas/label | Saling meniadakan | Saling meniadakan (mutually exclusive) | **Tidak** saling meniadakan, bisa muncul bersamaan |
| Ungkapan | "A **atau** B" | "A **atau** B **atau** C" | "A **dan** B **dan** C" |
| Contoh | Tumor ganas **atau** jinak; email spam **atau** bukan | Bunga Iris: Setosa **atau** Versicolor **atau** Virginica | Satu foto berlabel "pantai" **dan** "gunung"; satu lagu "happy" **dan** "relaxing" |
| Bentuk `y` | Satu kolom, 2 nilai unik, mis. `[0, 1, 1, 0]` | Satu kolom, >2 nilai unik, mis. `[0, 2, 1, 1]` | Matriks biner `(n_sampel x n_label)`, mis. `[[1,0,1],[0,1,1]]` |
| Aktivasi output | Sigmoid (1 neuron) | Softmax (jumlah probabilitas = 1), prediksi dengan `argmax` | Sigmoid per label (independen), lalu diberi threshold, mis. 0.5 |
| Loss yang lazim | Binary cross-entropy | Categorical cross-entropy | Binary cross-entropy per label |
| Metrik umum | Accuracy, precision, recall, F1, ROC-AUC | Accuracy, macro/micro/weighted F1, confusion matrix | Hamming loss, subset accuracy, F1 (micro/macro/samples) |
| Cara identifikasi dengan pandas | `df[label].nunique() == 2` | `df[label].nunique() > 2` | `df_label.sum(axis=1).max() > 1` (ada baris dengan label lebih dari 1) |

**Pembeda utama multi-class vs multi-label:** pada multi-class tiap data punya **tepat satu** kelas (ATAU), sedangkan pada multi-label tiap data bisa punya **beberapa label sekaligus** (DAN).

## 2. Persiapan

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)


def identifikasi_jenis(df_label):
    """
    Identifikasi jenis klasifikasi dari DataFrame label (menggunakan pandas).

    - 1 kolom label:
        2 nilai unik  -> binary
        >2 nilai unik -> multi-class
    - >1 kolom label (matriks 0/1):
        ada baris dengan jumlah label > 1 -> multi-label
        semua baris tepat 1 label         -> multi-class (one-hot)
    """
    if df_label.shape[1] == 1:
        n_kelas = df_label.iloc[:, 0].nunique()
        if n_kelas == 2:
            return "binary"
        if n_kelas > 2:
            return "multi-class"
        return "tidak valid (hanya 1 kelas)"

    nilai = set(pd.unique(df_label.to_numpy().ravel()).tolist())
    if not nilai.issubset({0, 1}):
        return "multi-output (bukan klasifikasi biasa)"
    if (df_label.sum(axis=1) > 1).any():
        return "multi-label"
    return "multi-class (one-hot)"


def _dense(a):
    return a.toarray() if sparse.issparse(a) else np.asarray(a)


# ----- loader: semuanya mengembalikan (df_fitur, df_label) -----
def load_sklearn(nama_fungsi, nama_label):
    """Dataset bawaan scikit-learn (salinan dari dataset UCI)."""
    from sklearn import datasets
    d = getattr(datasets, nama_fungsi)(as_frame=True)
    label = d.frame["target"].map(dict(enumerate(d.target_names))).rename(nama_label)
    return d.frame.drop(columns="target"), label.to_frame()


def load_uci(id_uci):
    """Dataset dari UCI ML Repository lewat paket ucimlrepo."""
    from ucimlrepo import fetch_ucirepo
    d = fetch_ucirepo(id=id_uci)
    return d.data.features, d.data.targets.iloc[:, [0]]


def load_mulan(nama):
    """
    Dataset multi-label (asal: repositori MULAN / Multi-Label Classification Dataset
    Repository), dimuat dari salinan CSV di Hugging Face. Kolom label diberi akhiran
    "__target" di file CSV-nya.
    """
    url = ("https://huggingface.co/datasets/imodels/multitask-tabular-datasets/"
           f"resolve/main/{nama}.csv")
    df = pd.read_csv(url)
    kolom_label = [c for c in df.columns if str(c).endswith("__target")]
    if not kolom_label:
        raise ValueError(f"Kolom label (__target) tidak ditemukan. Kolom contoh: {list(df.columns[:5])}")
    df_label = df[kolom_label].astype(int)
    df_label.columns = [str(c).replace("__target", "") for c in kolom_label]
    return df.drop(columns=kolom_label), df_label


HASIL = []  # menampung ringkasan semua dataset


def analisis(jenis_diharapkan, nama, loader):
    """Muat dataset sebagai DataFrame, identifikasi jenis klasifikasinya dengan pandas."""
    print(f"=== {nama} ===")
    try:
        df_fitur, df_label = loader()
    except Exception as e:
        print(f"Gagal memuat: {type(e).__name__}: {e}")
        print()
        return

    terdeteksi = identifikasi_jenis(df_label)
    n_label_kolom = df_label.shape[1]

    print(f"Ukuran fitur : {df_fitur.shape[0]} baris x {df_fitur.shape[1]} kolom")
    if n_label_kolom == 1:
        kolom_label = df_label.columns[0]
        n_kelas = df_label.iloc[:, 0].nunique()
        print(f"Kolom label  : {kolom_label}")
        print(f"Terdeteksi   : {terdeteksi} ({n_kelas} kelas unik)")
        print("Distribusi kelas:")
        print(df_label.iloc[:, 0].value_counts().to_string())
    else:
        kolom_label = f"{n_label_kolom} kolom biner"
        n_kelas = n_label_kolom
        jml = df_label.sum(axis=1)
        print(f"Kolom label  : {n_label_kolom} kolom biner ({', '.join(map(str, df_label.columns[:4]))}, ...)")
        print(f"Terdeteksi   : {terdeteksi}")
        print(f"Rata-rata label per sampel: {jml.mean():.2f} | maksimum: {jml.max()}")
        print("Jumlah sampel menurut banyaknya label aktif:")
        print(jml.value_counts().sort_index().to_string())
    print("5 baris pertama label:")
    print(df_label.head().to_string())
    print()

    HASIL.append({
        "Dataset": nama,
        "Jenis (diharapkan)": jenis_diharapkan,
        "Jenis (terdeteksi)": terdeteksi,
        "Sampel": df_fitur.shape[0],
        "Fitur": df_fitur.shape[1],
        "Kolom label": kolom_label,
        "Kelas/Label": n_kelas,
    })

## 3. Klasifikasi Binary

Pada klasifikasi binary hanya ada **2 kelas**, dan tiap data pasti masuk ke salah satunya (mis. ya/tidak, ganas/jinak, spam/bukan spam).

### 3.1 Breast Cancer Wisconsin (Diagnostic)

**Narasi:** Dataset ini berisi hasil pemeriksaan 569 massa payudara dengan metode *fine needle aspirate* (FNA). Dari citra digital sel yang diambil, dihitung 30 fitur numerik yang menggambarkan bentuk inti sel, seperti radius, tekstur, perimeter, dan luas. Setiap sampel diberi label diagnosis **malignant (ganas)** atau **benign (jinak)**, dengan komposisi sekitar 357 jinak dan 212 ganas. Karena hanya ada dua kemungkinan dan setiap pasien pasti masuk ke salah satunya, ini adalah contoh klasik klasifikasi **binary**.

- **Kolom label:** `diagnosis` (malignant / benign)
- **Asal dataset:** UCI Machine Learning Repository (id=17). Pembuat: William Wolberg, Nick Street, Olvi Mangasarian (University of Wisconsin), 1995.
- **Link:** https://archive.ics.uci.edu/dataset/17
- **DOI:** 10.24432/C5DW2B
- **Cara dimuat di sini:** `sklearn.datasets.load_breast_cancer` (salinan dataset UCI yang sama).

### 3.2 Spambase

**Narasi:** Spambase berisi 4601 email yang sudah ditandai oleh manusia sebagai **spam** atau **bukan spam**. Tiap email diwakili 57 fitur numerik, terutama frekuensi kemunculan kata dan karakter tertentu (mis. kata "free" atau tanda "!") serta panjang rangkaian huruf kapital. Tujuannya mempelajari pola yang membedakan email spam dari email biasa. Hanya ada dua kelas, sehingga termasuk klasifikasi **binary**.

- **Kolom label:** `Class` (1 = spam, 0 = bukan spam)
- **Asal dataset:** UCI Machine Learning Repository (id=94). Pembuat: Mark Hopkins, Erik Reeber, George Forman, Jaap Suermondt (Hewlett-Packard Labs), 1999.
- **Link:** https://archive.ics.uci.edu/dataset/94
- **Cara dimuat di sini:** paket `ucimlrepo` (`fetch_ucirepo(id=94)`).

### 3.3 Bank Marketing

**Narasi:** Dataset ini berisi 45.211 catatan kampanye telemarketing sebuah bank di Portugal. Fiturnya mencakup data nasabah (usia, pekerjaan, status pernikahan, pendidikan), riwayat kontak, dan kondisi ekonomi. Label menunjukkan apakah nasabah akhirnya **berlangganan deposito berjangka (yes)** atau **tidak (no)**. Kelasnya tidak seimbang karena mayoritas nasabah menjawab "no" (sekitar 88%), sehingga dataset ini juga sering dipakai untuk latihan menangani data tidak seimbang. Jenisnya **binary**.

- **Kolom label:** `y` (yes / no)
- **Asal dataset:** UCI Machine Learning Repository (id=222). Pembuat: S. Moro, P. Rita, P. Cortez, 2012.
- **Link:** https://archive.ics.uci.edu/dataset/222
- **Cara dimuat di sini:** paket `ucimlrepo` (`fetch_ucirepo(id=222)`).

In [ ]:
analisis("binary", "Breast Cancer Wisconsin (Diagnostic)", lambda: load_sklearn("load_breast_cancer", "diagnosis"))
analisis("binary", "Spambase", lambda: load_uci(94))
analisis("binary", "Bank Marketing", lambda: load_uci(222))

## 4. Klasifikasi Multi-class (ATAU)

Pada multi-class ada **lebih dari 2 kelas**, tetapi tiap data hanya punya **satu** kelas (A **atau** B **atau** C).

### 4.1 Iris

**Narasi:** Iris adalah dataset klasik yang berisi 150 bunga iris dari 3 spesies, masing-masing 50 sampel. Tiap bunga diukur dengan 4 fitur, yaitu panjang dan lebar sepal serta panjang dan lebar petal (dalam cm). Satu bunga hanya bisa termasuk **satu** spesies, yaitu *setosa*, *versicolor*, **atau** *virginica*, sehingga ini adalah contoh paling sederhana klasifikasi **multi-class**.

- **Kolom label:** `species` (setosa / versicolor / virginica)
- **Asal dataset:** UCI Machine Learning Repository (id=53). Data asli dikumpulkan dan dipublikasikan oleh R.A. Fisher (1936).
- **Link:** https://archive.ics.uci.edu/dataset/53
- **Cara dimuat di sini:** `sklearn.datasets.load_iris`.

### 4.2 Wine

**Narasi:** Dataset Wine berisi 178 sampel wine dari wilayah yang sama di Italia, tetapi berasal dari **3 kultivar (cultivar) anggur** yang berbeda. Setiap sampel dianalisis secara kimia dan menghasilkan 13 fitur numerik, antara lain kadar alkohol, asam malat, magnesium, flavonoid, dan intensitas warna. Tugasnya menebak kultivar asal wine dari komposisi kimianya. Tiap wine hanya berasal dari satu kultivar, jadi jenisnya **multi-class**.

- **Kolom label:** `cultivar` (class_0 / class_1 / class_2)
- **Asal dataset:** UCI Machine Learning Repository (id=109). Donor: M. Forina dkk., Institute of Pharmaceutical and Food Analysis and Technologies, Genoa.
- **Link:** https://archive.ics.uci.edu/dataset/109
- **Cara dimuat di sini:** `sklearn.datasets.load_wine`.

### 4.3 Dry Bean

**Narasi:** Dry Bean berisi 13.611 butir kacang kering yang difoto, lalu dari citranya diekstrak 16 fitur bentuk dan ukuran (mis. area, perimeter, panjang sumbu, dan kebulatan). Kacang dikelompokkan ke dalam **7 jenis**: Barbunya, Bombay, Cali, Dermason, Horoz, Seker, dan Sira. Jumlah tiap kelas tidak sama; Dermason paling banyak (3.546) dan Bombay paling sedikit (522). Satu butir kacang hanya punya satu jenis, jadi ini **multi-class** dengan distribusi kelas yang tidak seimbang.

- **Kolom label:** `Class` (7 jenis kacang)
- **Asal dataset:** UCI Machine Learning Repository (id=602). Pembuat: Murat Koklu dan Ilker Ali Ozkan, 2020.
- **Link:** https://archive.ics.uci.edu/dataset/602
- **Cara dimuat di sini:** paket `ucimlrepo` (`fetch_ucirepo(id=602)`).

In [ ]:
analisis("multi-class", "Iris", lambda: load_sklearn("load_iris", "species"))
analisis("multi-class", "Wine", lambda: load_sklearn("load_wine", "cultivar"))
analisis("multi-class", "Dry Bean", lambda: load_uci(602))

## 5. Klasifikasi Multi-label (DAN)

Pada multi-label, tiap data boleh punya **banyak label sekaligus** (A **dan** B **dan** C), dan label tidak saling meniadakan. Label disimpan sebagai **kolom-kolom biner** (1 = label aktif, 0 = tidak).

Ketiga dataset di bagian ini berasal dari **MULAN Multi-label Dataset Repository** (http://mulan.sourceforge.net/datasets-mlc.html). Di notebook ini datanya dimuat dari **salinan CSV** yang dikonversi dari *Multi-Label Classification Dataset Repository* (Universidad de Córdoba, https://uco.es/kdis/mllresources), tersedia di Hugging Face: https://huggingface.co/datasets/imodels/multitask-tabular-datasets (file `yeast.csv`, `scene.csv`, `emotions.csv`).

### 5.1 Yeast

**Narasi:** Dataset ini berisi 2417 gen dari yeast *Saccharomyces cerevisiae*. Tiap gen diwakili 103 fitur yang berasal dari data ekspresi microarray dan profil filogenetik. Satu gen dapat menjalankan **beberapa fungsi biologis sekaligus**, sehingga tiap gen punya himpunan label dari 14 kategori fungsi. Rata-rata satu gen memiliki sekitar 4,2 label aktif, jadi ini jelas **multi-label**, bukan multi-class.

- **Kolom label:** 14 kolom biner (kategori fungsi gen)
- **Asal dataset:** MULAN repository. Referensi asli: A. Elisseeff dan J. Weston, *A kernel method for multi-labelled classification*, NIPS 14, 2001.
- **Link:** http://mulan.sourceforge.net/datasets-mlc.html

### 5.2 Scene

**Narasi:** Scene berisi 2407 citra pemandangan alam yang diwakili 294 fitur visual. Tiap citra dapat diberi label dari 6 kategori (mis. *Beach*, *Mountain*, *Field*, *Urban*), dan satu gambar bisa memuat lebih dari satu sekaligus, misalnya **pantai dan gunung** dalam satu foto. Namun, citra dengan banyak label relatif jarang; rata-rata hanya sekitar 1,07 label per citra. Tetap saja, karena ada citra dengan lebih dari satu label, jenisnya **multi-label**.

- **Kolom label:** 6 kolom biner (kategori pemandangan)
- **Asal dataset:** MULAN repository. Referensi asli: M.R. Boutell, J. Luo, X. Shen, C.M. Brown, *Learning multi-label scene classification*, Pattern Recognition 37(9), 2004.
- **Link:** http://mulan.sourceforge.net/datasets-mlc.html

### 5.3 Emotions

**Narasi:** Emotions berisi 593 lagu yang diwakili 72 fitur audio. Tiap lagu dilabeli emosi yang ditimbulkannya berdasarkan model emosi Tellegen-Watson-Clark, dengan 6 kategori emosi (seperti *happy-pleased*, *relaxing-calm*, *sad-lonely*, dan *angry-aggressive*). Satu lagu bisa membangkitkan beberapa emosi sekaligus, misalnya **happy dan relaxing**, dengan rata-rata sekitar 1,87 label per lagu. Karena itu jenisnya **multi-label**.

- **Kolom label:** 6 kolom biner (kategori emosi)
- **Asal dataset:** MULAN repository. Referensi asli: K. Trohidis, G. Tsoumakas, G. Kalliris, I. Vlahavas, *Multilabel classification of music into emotions*, ISMIR 2008.
- **Link:** http://mulan.sourceforge.net/datasets-mlc.html

In [ ]:
analisis("multi-label", "Yeast", lambda: load_mulan("yeast"))
analisis("multi-label", "Scene", lambda: load_mulan("scene"))
analisis("multi-label", "Emotions", lambda: load_mulan("emotions"))

## 6. Ringkasan Hasil Identifikasi

Tabel di bawah dibuat otomatis dari hasil menjalankan cell-cell di atas. Kolom **Jenis (terdeteksi)** harus sama dengan **Jenis (diharapkan)**.

In [ ]:
df_ringkasan = pd.DataFrame(HASIL)
df_ringkasan["Cocok?"] = np.where(
    df_ringkasan["Jenis (diharapkan)"] == df_ringkasan["Jenis (terdeteksi)"], "Ya", "Tidak"
)
df_ringkasan

## 7. Kesimpulan

- **Binary**: kolom label punya **2 nilai unik** (`nunique() == 2`).
- **Multi-class**: kolom label punya **lebih dari 2 nilai unik**, dan tiap baris punya **tepat satu** kelas.
- **Multi-label**: label berupa **beberapa kolom biner**, dan ada baris yang jumlah labelnya **lebih dari 1** (`sum(axis=1).max() > 1`). Inilah pembeda utamanya dari multi-class, yang tiap barisnya hanya punya satu kelas.